# Steam Scrapper

GitHub reference: https://github.com/mmmmmm44/steam_scraping_api/tree/main  
Steam API documentation: https://partner.steamgames.com/doc/store/getreviews


Games with initially bad to good reviews
*   Cyberpunk 2077 (steam id: 1091500, ~344,977 reviews)
*  Sea of Thieves (steam id: 1172620, ~154,115 reviews)
* Fallout 76 (steam id: 1151340, ~69,565 reviews)
* Total War: Rome 2  (steam id: 214950, ~48,045 reviews)
  *(total war and rome 2 in the reddit post are the same game)*
* Days Gone (steam id: 1259420, ~42,657 reviews)
* Wildfrost (steam id: 1811990, ~5,219 reviews)
* Final Fantasy 14 (steam id: 39210, ~61,395 reviews)
* Warhammer 40k: Darktide (steam id: 1361210, ~75,584 reviews)
*  Battlefield 2042 (steam id: 1517290, ~120,270 reviews)
* Wasteland 3 (replaced a random wastelanders)

For the params:
```
params = {
    'json': 1,
    'language': 'english',
    'cursor': '*',                                  # set the cursor to retrieve reviews from a specific "page"
    'num_per_page': 100,
    'filter': 'recent',
    'review_type': 'all',
    'purchase_type': 'all'
}
```

In [ ]:
import requests
from datetime import datetime,timezone
import pandas as pd

UTC = timezone.utc


## Functions



In [3]:
def get_user_reviews(review_appid, params):

    user_review_url = f'https://store.steampowered.com/appreviews/{review_appid}'
    req_user_review = requests.get(
        user_review_url,
        params=params
    )

    if req_user_review.status_code != 200:
        print(f'Fail to get response. Status code: {req_user_review.status_code}')
        return {"success": 2}

    try:
        user_reviews = req_user_review.json()
    except ValueError:
        print("Failed to decode JSON response.")
        return {"success": 2}

    return user_reviews

In [4]:
# while (not passed_start_time or not passed_end_time):
def loop_through_reviews(review_appid, params):
  selected_reviews = []
  counter = 0
  while (True):
      # pass app id and params here to get the review response
      reviews_response = get_user_reviews(review_appid, params)

      # check if the review response is a success
      # not success?
      if reviews_response["success"] != 1:
          print("Not a success")
          print(reviews_response)
          break

      if reviews_response["query_summary"]['num_reviews'] == 0:
          print("No reviews.")
          print(reviews_response)
          break
      # enf of checking

      for review in reviews_response["reviews"]:
          recommendation_id = review['recommendationid']

          timestamp_created = review['timestamp_created'] # these are unix timestamps
          timestamp_updated = review['timestamp_updated']

          # extract the useful (to me) data
          author_steamid = review['author']['steamid']        # will automatically redirect to the profileURL if any
          playtime_forever = review['author']['playtime_forever']
          playtime_last_two_weeks = review['author']['playtime_last_two_weeks']
          playtime_at_review_minutes = 0
          try:
            playtime_at_review_minutes = review['author']['playtime_at_review']
          except Exception:
            playtime_at_review_minutes = 0
          last_played = review['author']['last_played']

          review_text = review['review']
          voted_up = review['voted_up']
          votes_up = review['votes_up']
          votes_funny = review['votes_funny']
          weighted_vote_score = review['weighted_vote_score']
          steam_purchase = review['steam_purchase']
          received_for_free = review['received_for_free']
          written_during_early_access = review['written_during_early_access']

          my_review_dict = {
              'recommendationid': recommendation_id,
              'author_steamid': author_steamid,
              'playtime_at_review_minutes': playtime_at_review_minutes,
              'playtime_forever_minutes': playtime_forever,
              'playtime_last_two_weeks_minutes': playtime_last_two_weeks,
              'last_played': last_played,

              'review_text': review_text,
              'timestamp_created': datetime.fromtimestamp(timestamp_created,UTC).strftime('%Y-%m-%d %H:%M:%S UTC'),
              'timestamp_updated': datetime.fromtimestamp(timestamp_updated,UTC).strftime('%Y-%m-%d %H:%M:%S UTC'),

              'voted_up': voted_up,
              'votes_up': votes_up,
              'votes_funny': votes_funny,
              'weighted_vote_score': weighted_vote_score,
              'steam_purchase': steam_purchase,
              'received_for_free': received_for_free,
              'written_during_early_access': written_during_early_access,
          }

          selected_reviews.append(my_review_dict)

      # go to next page
      try:
          cursor = reviews_response['cursor']         # cursor field does not exist in the last page
      except Exception:
          cursor = ''

      # no next page
      # EXIT the true loop
      if not cursor:
          print("Reached the end of all comments.")
          # print(counter, "pages")
          break

      # set the cursor object to move to next page to continue
      params['cursor'] = cursor
      counter += 1
      print('To next page. Next page cursor:', cursor)

    # after all the pages have been checked
  return selected_reviews


In [ ]:
game_appid_map = {    
    "cyberpunk_2077": 1091500, # CyberPunk 2077
    "sea_of_thieves": 1172620, # Sea of Thieves
    "fallout_76": 1151340, # Fallout 76
    "total_war_rome_ii": 214950,  # Total War: Rome 2
    "days_gone": 1259420, # Days Gone
    "wildfrost":1811990, # Wildfrost
    "final_fantasy_14":39210,   # Final Fantasy 14
    "wasteland_3":1361210, # Warhammer 40k: Darktide
    "battlefield_2042":1517290, # Battlefield 2
    "warhammer":719040, # Wasteland 3
}

params = {
        'json':1,
        'language': 'english',
        'cursor': '*',                                  # set the cursor to retrieve reviews from a specific "page"
        'num_per_page': 100,
        'filter': 'recent',
        'review_type': 'all',
        'purchase_type': 'all'
    }

filter_off_topic_activity_params = {
    """Used for total_war_rome_ii, days_gone"""
        'json':1,
        'language': 'english',
        'cursor': '*',                                  # set the cursor to retrieve reviews from a specific "page"
        'num_per_page': 100,
        'filter': 'recent',
        'review_type': 'all',
        'purchase_type': 'all',
        'filter_offtopic_activity': 0
    }

## Execution

In [6]:
for game,review_appid in game_appid_map.items():
    print("Processing reviews for game:", game)
    # get the reviews
    if game in ["total_war_rome_ii", "days_gone"]:
        selected_reviews = loop_through_reviews(review_appid, filter_off_topic_activity_params)
        print("Using filter_offtopic_activity")
    else:
        selected_reviews = loop_through_reviews(review_appid, params)

    # save the reviews to a csv file
    df = pd.DataFrame(selected_reviews)
    df.to_csv(f'./{game}_review.csv', index=False)

Processing reviews for game: cyberpunk_2077
To next page. Next page cursor: AoJw5KOq6pUDeK7M2QU=
To next page. Next page cursor: AoJwo9nB6JUDf8qx2QU=
To next page. Next page cursor: AoJ449GB55UDer2b2QU=
To next page. Next page cursor: AoJw6umh5ZUDdbWE2QU=
To next page. Next page cursor: AoJwgZ3M45UDfqvp2AU=
To next page. Next page cursor: AoJ4uIK94pUDccTb2AU=
To next page. Next page cursor: AoJ4782T4ZUDe+q+2AU=
To next page. Next page cursor: AoJ4n5j535UDe8is2AU=
To next page. Next page cursor: AoJwnNPG3pUDcJKQ2AU=
To next page. Next page cursor: AoJ46KaR3ZUDcrf61wU=
To next page. Next page cursor: AoJw7+fN25UDcZ7h1wU=
To next page. Next page cursor: AoJ4//2y2pUDcKvJ1wU=
To next page. Next page cursor: AoJwyIDT2JUDcdeu1wU=
To next page. Next page cursor: AoJw+t+V15UDevWU1wU=
To next page. Next page cursor: AoJww7u71ZUDc7L+1gU=
To next page. Next page cursor: AoJ4o+nK05UDdM7e1gU=
To next page. Next page cursor: AoJ4zNGL0pUDe5rD1gU=
To next page. Next page cursor: AoJ49bDP0JUDeuuu1gU=
To

KeyboardInterrupt: 